# Этап 5: сравнение IrResnet4 (hidden=72)

**Цель:** E1–E4 — SMARTS-only vs SMARTS+peak × контекст on/off.

**Вход:** датасет с `labels_structure_smarts.parquet` (лучше `dataset_v003`; допускается v002).

**Выход:** `runs/exp_v003/summary.json`, bar chart F1.

## Выбор среды (выполните ОДНУ ячейку)

| Среда | Запустить | Пропустить |
|-------|-----------|------------|
| **Google Colab** | **A. Colab** (+ при полном датасете **A2. Drive**) | **B. Local** |
| **Локальный Jupyter** | **B. Local** | **A** и **A2** (включая `drive.mount`) |

После A или B выполните **C. Пути и данные**.
Подробнее: [`docs/NOTEBOOKS.md`](../docs/NOTEBOOKS.md).


In [ ]:
# === A. Colab: окружение ===
# Локально эту ячейку НЕ запускайте (см. B. Local).
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Lamblador/IR_expert_system_3.git'
REPO_BRANCH = 'colab-v1'
REPO_DIR = Path('/content/IR_expert_system_3')

def _run_git(cmd, cwd=None):
    print('git', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if (REPO_DIR / '.git').is_dir():
    _run_git(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_DIR)
    _run_git(['git', 'checkout', REPO_BRANCH], cwd=REPO_DIR)
    _run_git(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=REPO_DIR)
else:
    if REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} существует, но это не git-репозиторий')
    _run_git([
        'git', 'clone', '-b', REPO_BRANCH, '--single-branch',
        REPO_URL, str(REPO_DIR),
    ])

ROOT = REPO_DIR.resolve()
os.chdir(ROOT)
try:
    from IPython import get_ipython
    get_ipython().run_line_magic('cd', str(ROOT))
except Exception:
    pass
rev = subprocess.check_output(
    ['git', 'rev-parse', '--short', 'HEAD'], cwd=ROOT, text=True
).strip()
print(f'ROOT: {ROOT} @ {REPO_BRANCH} ({rev})')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[torch]'], check=True)
print('pip install OK')
IR_ENV = 'colab'


### A2. Google Drive (только Colab full)

Пропустите для HF smoke и локально.

In [ ]:
# === A2. Colab Drive (full dataset) ===
# Нужен только для полного датасета на Google Drive.
# Для HF smoke (dataset_mini) эту ячейку ПРОПУСТИТЕ.
# Локально НЕ запускайте.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
IR_DATA = Path('/content/drive/MyDrive/ir_data')
RUNS_DRIVE = Path('/content/drive/MyDrive/ir_expert_system_3/runs')
RUNS_DRIVE.mkdir(parents=True, exist_ok=True)
print('IR_DATA exists:', IR_DATA.exists(), IR_DATA)
print('RUNS_DRIVE:', RUNS_DRIVE)


### B. Локальный Jupyter

Пропустите в Colab.

In [ ]:
# === B. Local: окружение ===
# В Google Colab эту ячейку НЕ запускайте (см. A. Colab).
import os
import subprocess
import sys
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    """Walk-up до каталога с pyproject.toml (фикс nested-clone из notebooks/)."""
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / 'pyproject.toml').is_file():
            return p
    raise FileNotFoundError(
        'Не найден pyproject.toml выше cwd. '
        'Откройте ноутбук из клона репозитория или cd в корень IR_expert_system_3.'
    )

ROOT = _find_repo_root(Path.cwd())
os.chdir(ROOT)
try:
    from IPython import get_ipython
    get_ipython().run_line_magic('cd', str(ROOT))
except Exception:
    pass
print('ROOT (local, ветку не переключаем):', ROOT)

FORCE_REINSTALL = False  # True — принудительно pip install -e .[torch]
need_install = FORCE_REINSTALL
if not need_install:
    try:
        import ir_pipeline  # noqa: F401
    except ImportError:
        need_install = True
if need_install:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[torch]'],
        check=True,
    )
    print('pip install OK')
else:
    print('ir_pipeline уже установлен — pip пропущен (FORCE_REINSTALL=True для переустановки)')
IR_ENV = 'local'


### C. Пути и данные

После A или B.

In [ ]:
# === C. Пути и данные ===
# Выполните после A или B. Контракт: ROOT, PATHS_YAML, paths, DATASET_DIR, BANDS_YAML, RUNS_DIR
import os
from pathlib import Path
from ir_pipeline.config_loader import load_yaml, resolve_paths

SMOKE_VERSION = 'dataset_mini'
FULL_VERSION = 'dataset_v003'
# Режим данных (если IR_ENV не задан — auto):
#   'local'       — configs/paths.local.yaml
#   'colab_smoke' — HF mini, paths.huggingface.yaml
#   'colab_full'  — Drive, paths.colab.yaml
DATA_MODE = 'auto'  # или 'local' | 'colab_smoke' | 'colab_full'

if 'IR_ENV' not in globals():
    IR_ENV = 'colab' if Path('/content').exists() else 'local'

if DATA_MODE == 'auto':
    if IR_ENV == 'local':
        DATA_MODE = 'local'
    elif 'IR_DATA' in globals() and Path(IR_DATA).exists():
        DATA_MODE = 'colab_full'
    else:
        DATA_MODE = 'colab_smoke'

if DATA_MODE == 'local':
    PATHS_YAML = Path('configs/paths.local.yaml')
    if not PATHS_YAML.is_file():
        raise FileNotFoundError(
            'Нет configs/paths.local.yaml — скопируйте configs/paths.local.example.yaml '
            'и пропишите raw_jcamp_dir / processed_root.'
        )
elif DATA_MODE == 'colab_full':
    if 'IR_DATA' not in globals():
        raise RuntimeError('Colab full: сначала выполните A2 (Drive mount) → IR_DATA')
    os.environ['IR_PROCESSED_ROOT'] = str(Path(IR_DATA) / 'processed')
    PATHS_YAML = Path('configs/paths.colab.yaml')
elif DATA_MODE == 'colab_smoke':
    PATHS_YAML = Path('configs/paths.huggingface.yaml')
else:
    raise ValueError(f'Неизвестный DATA_MODE={DATA_MODE!r}')

paths_cfg = load_yaml(PATHS_YAML)
if DATA_MODE == 'colab_smoke':
    paths_cfg['dataset_version'] = SMOKE_VERSION
    paths_cfg.pop('dataset_profile', None)
elif DATA_MODE == 'colab_full':
    paths_cfg['dataset_version'] = paths_cfg.get('dataset_version') or FULL_VERSION
# local: dataset_version / profile из yaml

paths = resolve_paths(paths_cfg)
DATASET_DIR = paths['processed_root'] / str(paths['dataset_version'])
BANDS_YAML = paths['bands_config']
RUNS_DIR = Path('runs')
RUNS_DIR.mkdir(parents=True, exist_ok=True)

spectra = DATASET_DIR / 'spectra.npz'
if not spectra.is_file():
    if DATA_MODE == 'colab_smoke':
        print(f'{DATASET_DIR} нет → fetch HF {SMOKE_VERSION}.zip')
        !ir-pipeline fetch-data --filename dataset_mini.zip --extract-to data/processed
        if not spectra.is_file():
            raise FileNotFoundError(f'После fetch нет {spectra}')
    else:
        raise FileNotFoundError(
            f'Нет {spectra}. Проверьте paths yaml / Drive / локальные каталоги. '
            f'DATA_MODE={DATA_MODE}, PATHS_YAML={PATHS_YAML}'
        )
print('DATA_MODE:', DATA_MODE)
print('PATHS_YAML:', PATHS_YAML)
print('DATASET_DIR:', DATASET_DIR)
print('dataset_version:', paths['dataset_version'])
print('raw_jcamp_dir:', paths['raw_jcamp_dir'])


In [ ]:
%matplotlib inline


In [ ]:
from pathlib import Path
import pandas as pd
from ir_pipeline.dataset_preview import build_multilabel_matrix
from ir_pipeline.resnet_input import load_model_inputs

DATASET = DATASET_DIR
_, _, spec_ids, _, _ = load_model_inputs(DATASET)
bands = BANDS_YAML
rows = []
for schema in ['structure_smarts', 'structure', 'spectrum']:
    try:
        Y, _ = build_multilabel_matrix(DATASET, spec_ids, bands, label_schema=schema)
        rows.append({'schema': schema, 'mean_labels': float(Y.sum(axis=1).mean()), 'positives': int(Y.sum())})
    except FileNotFoundError as e:
        rows.append({'schema': schema, 'error': str(e)})
pd.DataFrame(rows)


In [ ]:
from copy import deepcopy
from pathlib import Path
from ir_pipeline.config_loader import load_yaml, merge_train_defaults
from ir_pipeline.irresnet_train import train_irresnet_run

base_cfg = merge_train_defaults(load_yaml(Path('configs/train_irresnet_experiments.yaml')))
EXPERIMENTS = [
    ('E1', 'structure_smarts', False),
    ('E2', 'structure', False),
    ('E3', 'structure_smarts', True),
    ('E4', 'structure', True),
]
EXP_ROOT = RUNS_DIR / 'exp_v003'
summaries = []
for exp_id, schema, use_ctx in EXPERIMENTS:
    cfg = deepcopy(base_cfg)
    cfg['use_measurement_context'] = use_ctx
    run_dir = EXP_ROOT / f'{exp_id.lower()}_h72_{"ctx" if use_ctx else "noctx"}_{schema}'
    print('===', exp_id, schema, 'context=', use_ctx, '=>', run_dir)
    s = train_irresnet_run(
        dataset_dir=DATASET,
        run_dir=run_dir,
        bands_yaml=bands,
        train_cfg=cfg,
        label_schema=schema,
        use_measurement_context=use_ctx,
    )
    s['experiment'] = exp_id
    summaries.append(s)
pd.DataFrame(summaries)


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd

df = pd.DataFrame(summaries)
out = RUNS_DIR / 'exp_v003'
out.mkdir(parents=True, exist_ok=True)
(out / 'summary.json').write_text(df.to_json(orient='records', indent=2), encoding='utf-8')
fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(df))
ax.bar(x, df['test_f1_weighted'], color='steelblue')
ax.set_xticks(list(x))
ax.set_xticklabels(df['experiment'], rotation=0)
ax.set_ylabel('test F1 weighted')
ax.set_title('IrResnet4 experiments (hidden=72)')
fig.tight_layout()
fig.savefig(out / 'experiments_f1_weighted.png', dpi=140)
plt.show()
df
